# Load Forecast Result Charts

Generate the four repository charts for the final 2021 load-forecast result. The notebook reads artifacts from the public final-test script and writes landscape PNG files to `docs/forecasting/assets/`.

Before running, create the row-level artifact with:

```bash
python scripts/run_forecast_final_test.py --save-forecasts
```

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'src').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_DIR = PROJECT_ROOT / 'results' / 'forecasting'
ASSETS_DIR = PROJECT_ROOT / 'docs' / 'forecasting' / 'assets'
ASSETS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH = RESULTS_DIR / 'final_test_metrics.csv'
FORECASTS_PATH = RESULTS_DIR / 'final_test_forecasts.csv'

if not METRICS_PATH.exists() or not FORECASTS_PATH.exists():
    raise FileNotFoundError(
        'Run scripts/run_forecast_final_test.py --save-forecasts first.'
    )

metrics = pd.read_csv(METRICS_PATH)
forecasts = pd.read_csv(
    FORECASTS_PATH,
    parse_dates=['forecast_origin', 'forecast_timestamp'],
)
print(f'Loaded {len(metrics):,} metric rows and {len(forecasts):,} forecast rows.')

In [ ]:
COLORS = {
    'hgb': '#167D8D',
    'weekly': '#7A5195',
    'daily': '#8FA1B3',
    'actual': '#172B4D',
}
EXPERIMENT_LABELS = {
    'hgb_default_final_test_2021': 'HGB',
    'weekly_naive_final_test_2021': 'Weekly Naive',
    'daily_naive_final_test_2021': 'Daily Naive',
}
CHART_FONT = 'Arial'
TEXT_COLOR = '#172B4D'
MUTED_TEXT_COLOR = '#52616B'
GRID_COLOR = '#E6E9ED'

def save_figure(fig, filename, width=1400, height=800, margin=None):
    """Apply the repository chart style and export one PNG asset."""
    fig.update_layout(
        template='plotly_white',
        font=dict(family=CHART_FONT, size=20, color=TEXT_COLOR),
        title=dict(x=0.02, xanchor='left', font=dict(size=25)),
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=1.02,
            x=0,
            font=dict(size=20),
        ),
        margin=margin or dict(l=90, r=60, t=115, b=80),
        hovermode='x unified',
    )
    fig.update_xaxes(
        gridcolor=GRID_COLOR,
        title_font=dict(size=20),
        tickfont=dict(size=19),
    )
    fig.update_yaxes(
        gridcolor=GRID_COLOR,
        title_font=dict(size=20),
        tickfont=dict(size=19),
    )
    output_path = ASSETS_DIR / filename
    fig.write_image(output_path, width=width, height=height, scale=1)
    print(f'Saved {output_path.relative_to(PROJECT_ROOT)}')
    return fig

## 1. Overall model performance

In [ ]:
overall = metrics.loc[metrics['metric_scope'].eq('overall')].copy()
overall['label'] = overall['experiment_name'].map(EXPERIMENT_LABELS)
overall = overall.set_index('label').loc[
    ['Daily Naive', 'Weekly Naive', 'HGB']
].reset_index()
bar_colors = [COLORS['daily'], COLORS['weekly'], COLORS['hgb']]
bar_text = [
    f"{row.wape_percent:.2f}%"
    for row in overall.itertuples()
]

comparison_fig = go.Figure(
    go.Bar(
        x=overall['wape_percent'],
        y=overall['label'],
        orientation='h',
        marker_color=bar_colors,
        text=bar_text,
        textposition='outside',
        textfont=dict(size=20),
        hovertemplate='%{y}: %{x:.2f} kWh<extra></extra>',
    )
)
comparison_fig.update_layout(
    title='Final 2021 Test: HGB Reduces WAPE by 38.8% vs Weekly Naive',
    xaxis_title='Weighted absolute percentage error (%)',
    yaxis_title='',
    showlegend=False,
)
comparison_fig.update_xaxes(range=[0, overall['wape_percent'].max() * 1.38])
comparison_fig = save_figure(
    comparison_fig,
    'load_forecast_model_comparison.png',
)
comparison_fig

## 2. Representative 24-hour forecast

Select the HGB origin whose aggregated 24-hour WAPE is closest to the median origin WAPE. This avoids choosing an unusually good example while accounting for different load levels. The chart compares HGB with both seasonal baselines at the same origin.

In [ ]:
hgb_forecasts = forecasts.loc[
    forecasts['experiment_name'].eq('hgb_default_final_test_2021')
].copy()
origin_errors = hgb_forecasts.groupby('forecast_origin').agg(
    absolute_error_kwh=('absolute_error_kwh', 'sum'),
    actual_kwh=('actual_kwh', lambda values: values.abs().sum()),
)
origin_wape = 100 * (
    origin_errors['absolute_error_kwh'] / origin_errors['actual_kwh']
)
median_origin_wape = origin_wape.median()
review_origin = (origin_wape - median_origin_wape).abs().idxmin()
review = (
    hgb_forecasts.loc[hgb_forecasts['forecast_origin'].eq(review_origin)]
    .sort_values('horizon_hours')
)
baseline_review = forecasts.loc[
    forecasts['forecast_origin'].eq(review_origin)
].copy()
review_wape = (
    100 * review['absolute_error_kwh'].sum() / review['actual_kwh'].abs().sum()
)
print(
    f'Representative origin: {review_origin}; '
    f'24-hour WAPE: {review_wape:.2f}%.'
)

example_fig = go.Figure()
example_fig.add_scatter(
    x=review['forecast_timestamp'],
    y=review['actual_kwh'],
    name='Actual load',
    mode='lines+markers',
    line=dict(color=COLORS['actual'], width=3),
    marker=dict(size=7),
)
for experiment_name, label, color, dash in (
    ('hgb_default_final_test_2021', 'HGB forecast', COLORS['hgb'], 'solid'),
    (
        'weekly_naive_final_test_2021',
        'Weekly Naive',
        COLORS['weekly'],
        'dash',
    ),
    (
        'daily_naive_final_test_2021',
        'Daily Naive',
        COLORS['daily'],
        'dot',
    ),
):
    model_review = baseline_review.loc[
        baseline_review['experiment_name'].eq(experiment_name)
    ].sort_values('horizon_hours')
    example_fig.add_scatter(
        x=model_review['forecast_timestamp'],
        y=model_review['prediction_kwh'],
        name=label,
        mode='lines',
        line=dict(color=color, width=3, dash=dash),
    )
example_fig.update_layout(
    title=(
        'Representative 24-Hour Forecast'
        f"<br><sup>Origin {review_origin:%Y-%m-%d %H:%M UTC} | "
        f'WAPE {review_wape:.2f}%</sup>'
    ),
    xaxis_title='Forecast timestamp (UTC)',
    yaxis_title='Gross load (kWh)',
)
example_fig = save_figure(
    example_fig,
    'load_forecast_representative_horizon.png',
    margin=dict(l=90, r=50, t=140, b=85),
)
example_fig

## 3. Error by month and local hour

In [ ]:
local_targets = hgb_forecasts['forecast_timestamp'].dt.tz_convert(
    'Europe/Berlin'
)
hgb_forecasts['target_local_month'] = local_targets.dt.month
hgb_forecasts['target_local_hour'] = local_targets.dt.hour
heatmap_groups = hgb_forecasts.groupby(
    ['target_local_month', 'target_local_hour'],
    sort=True,
)
heatmap_data = (
    100
    * heatmap_groups['absolute_error_kwh'].sum()
    / heatmap_groups['actual_kwh'].apply(lambda values: values.abs().sum())
).unstack('target_local_hour')
month_labels = [
    'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec',
]
heatmap_fig = go.Figure(
    go.Heatmap(
        z=heatmap_data.to_numpy(),
        x=heatmap_data.columns,
        y=month_labels,
        colorscale='YlOrRd',
        colorbar=dict(title='WAPE<br>(%)'),
        hovertemplate=(
            'Month: %{y}<br>Local hour: %{x}:00<br>'
            'WAPE: %{z:.2f}%<extra></extra>'
        ),
    )
)
heatmap_fig.update_layout(
    title='HGB Error Concentrates During Summer Working Hours',
    xaxis_title='Target local hour',
    yaxis_title='',
)
heatmap_fig.update_xaxes(dtick=1, showgrid=False)
heatmap_fig.update_yaxes(showgrid=False, autorange='reversed')
heatmap_fig = save_figure(
    heatmap_fig,
    'load_forecast_error_heatmap.png',
    height=850,
)
heatmap_fig

## 4. Error by forecast horizon

In [ ]:
horizon_metrics = metrics.loc[
    metrics['metric_scope'].eq('horizon')
].copy()
horizon_fig = go.Figure()
for experiment_name, label, color, dash in (
    ('hgb_default_final_test_2021', 'HGB', COLORS['hgb'], 'solid'),
    ('weekly_naive_final_test_2021', 'Weekly Naive', COLORS['weekly'], 'dash'),
    ('daily_naive_final_test_2021', 'Daily Naive', COLORS['daily'], 'dot'),
):
    model_metrics = horizon_metrics.loc[
        horizon_metrics['experiment_name'].eq(experiment_name)
    ].sort_values('horizon_hours')
    horizon_fig.add_scatter(
        x=model_metrics['horizon_hours'],
        y=model_metrics['wape_percent'],
        name=label,
        mode='lines+markers',
        line=dict(color=color, width=3, dash=dash),
        marker=dict(size=7),
    )

horizon_fig.update_layout(
    title='HGB Remains Better Across the Complete 24-Hour Horizon',
    xaxis_title='Forecast horizon (hours)',
    yaxis_title='Weighted absolute percentage error (%)',
)
horizon_fig.update_xaxes(dtick=1)
horizon_fig = save_figure(
    horizon_fig,
    'load_forecast_horizon_error.png',
)
horizon_fig